# Data Cleaner
This file turns ../data/real-estate.csv into ../data/real-estate-cleaned.csv


In [3]:
import pandas as pd
import numpy as np

# Searching for (latitude, longitude) by town. pip install geopy
from geopy.geocoders import Nominatim  
import time

# Saving town_names in a JSON file so the 169 fetched town locations are saved 
import json
import os

In [4]:
df = pd.read_csv("../data/processed/real-estate.csv")
print(df.head())
print()
print(df.shape)
print()
print(df.columns)

   List Year Date Recorded        Town Assessed Value  Sale Amount  \
0       2010    10/02/2010     Norwalk    $339,640.00  $265,000.00   
1       2010    10/02/2010   Greenwich    $450,000.00  $650,000.00   
2       2010    10/03/2010     Milford    $674,350.00  $788,000.00   
3       2010    10/04/2010  Bridgeport    $132,250.00  $148,000.00   
4       2010    10/04/2010     Bristol     $99,610.00   $32,000.00   

   Property Type Residential Type                    Location  
0  Single Family    Single Family                         NaN  
1            NaN              NaN                         NaN  
2  Single Family    Single Family                         NaN  
3  Single Family    Single Family  POINT (-73.21591 41.20061)  
4  Single Family    Single Family                         NaN  

(652402, 8)

Index(['List Year', 'Date Recorded', 'Town', 'Assessed Value', 'Sale Amount',
       'Property Type', 'Residential Type', 'Location'],
      dtype='str')


In [ ]:
""" 
We need every CSV entry to have a valid (longitude, latitude). 
We will use the coordinates of the center of the town. 
Ended up using this for the 169 towns of Connecticut: 
    https://www.mapsofworld.com/usa/states/connecticut/lat-long.html
"""

# Extract all town names from the CSV
town_names = dict()
for town in df["Town"].dropna().unique():  # dropna() removes missing values
    town_names[town] = None
print(f"The data references {len(town_names)} unique towns:\n")
for town in sorted(town_names.keys()):
    print(town)

# Calculate the (longitude, latitude) for the center of each town
# Save town_names (a dictionary of tuples) in a JSON file (tuples
#   become lists) so we don't need to send 169 requests to Nominatim 
#   every time the notebook runs
filename = "town_coords.json"
if os.path.exists(filename):
    # Turn the json into a dictionary of tuples
    with open(filename, "r") as f:
        coords = json.load(f)
    for town in town_names:
        if town in coords:
            town_names[town] = tuple(coords[town])  # town_names = {town: (long, lat)}
else:
    # Use the API to get all the values
    print("Fetching coordinates from Nominatim...")
    geolocator = Nominatim(user_agent="ri_real_estate_model")
    for town in town_names:
        print(f"About to locate \"{town}\"")
        location = geolocator.geocode(f"{town}, Connecticut, USA")
        if location:
            town_names[town] = (location.longitude, location.latitude)
        else:
            town_names[town] = None
        time.sleep(2)  # The API has a strict rate limit of 1 request per second
    # Write all the town names into a JSON file (tuples become lists)
    with open(filename, "w") as f:  
        json.dump(town_names, f)  

The data references 169 unique towns:

Andover
Ansonia
Ashford
Avon
Barkhamsted
Beacon Falls
Berlin
Bethany
Bethel
Bethlehem
Bloomfield
Bolton
Bozrah
Branford
Bridgeport
Bridgewater
Bristol
Brookfield
Brooklyn
Burlington
Canaan
Canterbury
Canton
Chaplin
Cheshire
Chester
Clinton
Colchester
Colebrook
Columbia
Cornwall
Coventry
Cromwell
Danbury
Darien
Deep River
Derby
Durham
East Granby
East Haddam
East Hampton
East Hartford
East Haven
East Lyme
East Windsor
Eastford
Easton
Ellington
Enfield
Essex
Fairfield
Farmington
Franklin
Glastonbury
Goshen
Granby
Greenwich
Griswold
Groton
Guilford
Haddam
Hamden
Hampton
Hartford
Hartland
Harwinton
Hebron
Kent
Killingly
Killingworth
Lebanon
Ledyard
Lisbon
Litchfield
Lyme
Madison
Manchester
Mansfield
Marlborough
Meriden
Middlebury
Middlefield
Middletown
Milford
Monroe
Montville
Morris
Naugatuck
New Britain
New Canaan
New Fairfield
New Hartford
New Haven
New London
New Milford
Newington
Newtown
Norfolk
North Branford
North Canaan
North Haven
North Stoni

In [ ]:
################################################################################
### Make sure the columns (Property Type, Residential Type) as (PT, RT) are 
###   consistent between new and old data entries. Use the newer, better version. 
### All properties with a null PT are removed (confuses the model).
### All properties where PT = "Commercial" or "Vacant Land" stay the same
###   because their standards never changed. 
### Residential properties keep their RT value (standard did not change),
###   but their PT values will be brought to standard
################################################################################
# Delete rows without a PT 
df["Property Type"] = df["Property Type"].str.strip()
df = df[df["Property Type"].notna() & (df["Property Type"] != "")]

# Formatting updates
def normalize_types(row):
    prop = row["Property Type"]
    res = row["Residential Type"]
    if prop in ["Commercial", "Vacant Land"]:
        return prop, res
    # If residential type exists, standardize to Residential
    if pd.notna(res):
        return "Residential", res
	# Return the values an ideally-formatted csv would contain
    return prop, res

# Call the update function on every row. Each function call receives a Series
#   representing the row, not just a single column like with .map()
df[["Property Type", "Residential Type"]] = df.apply(
    normalize_types, axis=1, result_type="expand"
)



################################################################################
### Add proper coordinates to each record, based on the town name
################################################################################
# Take a town name and return a csv-ready coordinate for it 
def town_to_point(town):
	lon, lat = town_names[town]
	return f"POINT ({lon} {lat})"
# Update the dataframe's entire "Location" column by calling town_to_point() 
#   on every record's "Location" value
df["Location"] = df["Town"].map(town_to_point) 



################################################################################
### Change the prices from strings (in the csv) into floats
################################################################################
# Remove money signs and commas (replace them with "")
for col in ["Sale Amount", "Assessed Value"]:
    df[col] = (
        df[col]
        .astype(str)
        .str.replace(r"[\$,]", "", regex=True)
        .str.strip()
    )
    df[col] = pd.to_numeric(df[col])



################################################################################
### Write the updated dataframe into the csv file 
################################################################################
df.to_csv("../data/real-estate-cleaned.csv", index=False)
